This notebook is for evaluating LexGlue. We are going to evaluate all of LexGlUE's datasets, including:


1.   SCOTUS
2.   ECtHR A
3.   ECtHR B
4.   LEDGAR
5.   EURLEX
6.   UNFAIR-ToS
7.   CaseHOLD

We are going to examine the accuracy (μ-F1 and m-F1) of RLM with this benchmark, and compare it with vanilla LLM.



In [ ]:
# package instalizations

%pip install rlms
%pip install datasets
%pip install scikit-learn
%pip install -U google-genai


In [ ]:
# imports

import os
from rlm import RLM
from datasets import load_dataset
from collections.abc import Sequence

In [ ]:
# Different LLMs API Keys

os.environ['OPENROUTER_API_KEY'] = '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'
os.environ['GEMINI_API_KEY'] = 'AQ.Ab8RN6L0opJavYC3imR-sg2dXBr8YK8rSVFjDSXz-cw3f97i7Q'
os.environ['GROQ_API_KEY'] = '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'

In [ ]:
# RLM Object instantiation

rlm = RLM(
    backend="gemini",
    backend_kwargs={"model_name": "gemini-2.5-flash"},
    verbose=True,  # Keep this True! It lets you watch the RLM write and execute Python code in real-time.
)

We define a function for prompt generation, based on the **Task Type** of these datasets. it can be:


*   multi-**class**-classification : 1 document → exactly 1 label
*   multi-**label**-classification : 1 document → 1 or more labels
*   multiple-choice QA : 1 document + n possible holdings → only 1 holding is correct



In [ ]:
TASK_TYPE: dict[int] = {
    "MULTI_CLASS": 1,
    "MULTI_LABEL": 2,
    "MULTIPLE_CHOICE_QA": 3
}


In [ ]:
# extract datasets labels based on task type
def extract_labels(task_type: int, dataset_features) -> Sequence[str]:
  if task_type == TASK_TYPE["MULTI_CLASS"]:
    return dataset_features['label'].names

  if task_type == TASK_TYPE["MULTI_LABEL"]:
    return dataset_features['labels'].feature.names

  else:
    raise KeyError(f"The following task type is invalid ({task_type}). It should be among this list: {TASK_TYPE.values()}")


In [ ]:
# 5. EURLEX

dataset_eurlex = load_dataset("coastalcph/lex_glue", "eurlex")
dataset_label_ids = extract_labels(2, dataset_eurlex['validation'].features)
print(extract_labels(2, dataset_eurlex['validation'].features))

In [ ]:
# for EURLEX, we need to get and build the dictionary and list of EuroVoc

import requests

EUROVOC_URL = (
    "https://raw.githubusercontent.com/"
    "nlpaueb/multi-eurlex/master/data/eurovoc_descriptors.json"
)

# Download and return the EuroVoc descriptors.
def load_eurovoc() -> dict:
    response = requests.get(EUROVOC_URL)
    response.raise_for_status()
    return response.json()


# return the mapping between vocab code and english equivalent
def get_eurovoc_label_map(dataset_label_ids) -> dict[str, str]:
    data = load_eurovoc()
    return {
        label_id: data[label_id]["en"]
        for label_id in dataset_label_ids
    }


# return the list of EuroVoc list (i.e EuroLex label names)
def get_eurovoc_label_names(dataset_label_ids) -> list[str]:
    label_map = get_eurovoc_label_map(dataset_label_ids)
    return [label_map[label_id] for label_id in dataset_label_ids]

eurolex_label_names = get_eurovoc_label_names(dataset_label_ids)
print(len(eurolex_label_names))

In [ ]:
# label names are imported from the offocial github repo for evaluating
# of one-shot test of lexGlue + llm (https://github.com/coastalcph/zeroshot_lexglue.git)

LEXGLUE_TASKS_DICT: dict[dict[str]] = {
    "SCOTUS": {
        "task_type": 1,
        "task_description": "Classify the United States Supreme Court opinion according to its primary legal issue area.",
        "label_names": ['Criminal Procedure', 'Civil Rights', 'First Amendment', 'Due Process', 'Privacy', 'Attorneys',
                'Unions', 'Economic Activity', 'Judicial Power', 'Federalism', 'Interstate Relations',
                'Federal Taxation', 'Miscellaneous']
    },
    "ECtHR_A": {
        "task_type": 2,
        "task_description": "Predict which Articles of the European Convention on Human Rights the applicant alleges were violated.",
        "label_names": ['Article 2 - Right to life', 'Article 3 - Prohibition of torture',  'Article 5 - Right to liberty and security',
                        'Article 6 - Right to a fair trial',  'Article 8 - Right to respect for private and family life',
                        'Article 9 - Freedom of thought, conscience and religion', 'Article 10 - Freedom of expression',
                        'Article 11 - Freedom of assembly and association', 'Article 14 - Prohibition of discriminatioN',
                        'Article P1-1 - Protection of property']
    },
    "ECtHR_B": {
        "task_type": 2,
        "task_description": "Predict which Articles of the European Convention on Human Rights the Court found to have been violated.",
        "label_names": ['Article 2 - Right to life', 'Article 3 - Prohibition of torture',  'Article 5 - Right to liberty and security',
                        'Article 6 - Right to a fair trial',  'Article 8 - Right to respect for private and family life',
                        'Article 9 - Freedom of thought, conscience and religion', 'Article 10 - Freedom of expression',
                        'Article 11 - Freedom of assembly and association', 'Article 14 - Prohibition of discriminatioN',
                        'Article P1-1 - Protection of property']
    },
    "LEDGAR": {
        "task_type": 1,
        "task_description": "Classify the contract provision according to its clause category.",
        "label_names": None
    },
    "EURLEX": {
        "task_type": 2,
        "task_description": "Assign all applicable EuroVoc legal concepts to this European Union legal document.",
        "label_names": eurolex_label_names
    },
    "UNFAIR_ToS": {
        "task_type": 2,
        "task_description": "Identify all categories of potentially unfair contractual terms that apply to this Terms of Service sentence.",
        "label_names": None
    },
    "CaseHOLD": {
        "task_type": 3,
        "task_description": "",
        "label_names": None
    }
}

In [ ]:
def get_data_item_label_attr_name(task_type: int) -> str:
  if task_type == TASK_TYPE["MULTI_CLASS"] or task_type == TASK_TYPE["MULTIPLE_CHOICE_QA"]:
    return 'label'

  return 'labels'

In [ ]:
# Builds a mapping from dataset label -> human-readable label.
# only for the datasets with not-None value as label_name in LEXGLUE_TASKS_DICT

def build_label_map(
    dataset_labels: Sequence[str],
    readable_labels: Sequence[str],
) -> dict[str, str]:

    print(dataset_labels)
    print(readable_labels)


    if len(dataset_labels) != len(readable_labels):
        raise ValueError(
            f"Length mismatch: {len(dataset_labels)} != {len(readable_labels)}"
        )

    return dict(zip(dataset_labels, readable_labels))

In [ ]:
def multi_class_classification_prompt(task_name: str, dataset_features, data_item) -> str:

  task_description = LEXGLUE_TASKS_DICT[task_name]["task_description"]
  if (LEXGLUE_TASKS_DICT[task_name]["label_names"] == None):
    labels = extract_labels(task_type=TASK_TYPE["MULTI_CLASS"], dataset_features=dataset_features)
  else:
    labels = LEXGLUE_TASKS_DICT[task_name]["label_names"]

  return f"""
You are performing a multi-class legal document classification task.

Task: {task_description}

Choose exactly ONE label from the candidate labels below.

Candidate labels: {labels}

Output requirements:
- Return only the exact label name from the candidate labels.
- Do not add explanations, punctuation, or additional text.

Document:
<document>
{data_item['text']}
</document>

""".strip()

In [ ]:
def multi_label_classification_prompt(task_name: str, dataset_features, data_item) -> list[str]:

  task_description = LEXGLUE_TASKS_DICT[task_name]["task_description"]
  if (LEXGLUE_TASKS_DICT[task_name]["label_names"] == None):
    labels = extract_labels(task_type=TASK_TYPE["MULTI_LABEL"], dataset_features=dataset_features)
  else:
    labels = LEXGLUE_TASKS_DICT[task_name]["label_names"]

  return f"""
You are performing a multi-label legal document classification task.

Task: {task_description}

Select every applicable label from the candidate labels below.
Zero, one, or multiple labels may apply.

Candidate labels: {labels}

Output requirements:
- Return the applicable labels as a python list from the candidate labels.
- Do not add explanations or additional text.
- If no label applies, return NONE.

Document:
<document>
{data_item['text']}
</document>

""".strip()

In [324]:
def multiple_choice_QA_prompt(data_item) -> str:
  holdings = data_item['endings']

  return f"""
  You are given an excerpt from a judicial decision and five candidate holdings.

  Select the candidate holding that correctly completes or corresponds to the
  legal reasoning in the provided context.

  Return only the index of the corresponding holding.
  The valid outputs are: 0, 1, 2, 3, or 4.
  Do not provide an explanation.

  Legal context:
  <context>
  {data_item['context']}
  </context>

  Candidate holdings:
  {holdings}
  """.strip()


In [ ]:

def generate_prompt(task_name: str, data_features, data_item: str) -> str:

  if task_name not in LEXGLUE_TASKS_DICT:
    raise KeyError(f'Task name is invalid ({task_name}). supported tasks: {", ".join(LEXGLUE_TASKS_DICT)}')

  task_type = LEXGLUE_TASKS_DICT[task_name]['task_type']
  if task_type not in TASK_TYPE.values():
    raise KeyError(f'Task type is invalid ({task_type}). supported task types: {", ".join(TASK_TYPE)}')

  # Case 1: multi-class-classification
  if task_type == TASK_TYPE['MULTI_CLASS']:
    return multi_class_classification_prompt(task_name, data_features, data_item)

  # Case 2: multi-label-classification
  if task_type == TASK_TYPE["MULTI_LABEL"]:
    return multi_label_classification_prompt(task_name, data_features, data_item)
    # Case 3: multiple-choice QA
  return multiple_choice_QA_prompt(data_item)




- We are now working with the validation set, and then, work with the test set.

**--------------------**

**Multi Class Tasks:**

**--------------------**

In [ ]:
# 1. SCOTUS (US Supreme Court)

dataset_scotus = load_dataset("coastalcph/lex_glue", "scotus")
print(extract_labels(1, dataset_scotus['validation'].features))
print(len(dataset_scotus['validation']))

In [ ]:
# 2. LEDGAR

dataset_ledgar = load_dataset("coastalcph/lex_glue", "ledgar")
print(extract_labels(1, dataset_ledgar['validation'].features))
print(len(dataset_ledgar['validation']))

**--------------------**

**Multi Label Tasks:**

**--------------------**

In [ ]:
# 3. ECtHR A

dataset_ecthr_a = load_dataset("coastalcph/lex_glue", "ecthr_a")
print(extract_labels(2, dataset_ecthr_a['validation'].features))
print(len(dataset_ecthr_a['validation']))


In [ ]:
# Also EURLEX belongs here, but since we needed to get the EuroVoc labels, we have already done it above.

In [ ]:
# 4. ECtHR B

dataset_ecthr_b = load_dataset("coastalcph/lex_glue", "ecthr_b")
print(extract_labels(2, dataset_ecthr_b['validation'].features))
print(len(dataset_ecthr_b['validation']))

In [ ]:
# 6. UNFAIR-ToS

dataset_unfair_tos = load_dataset("coastalcph/lex_glue", "unfair_tos")
print(extract_labels(2, dataset_unfair_tos['validation'].features))
print(len(dataset_unfair_tos['validation']))

In [ ]:
# 7. Case Hold

dataset_casehold = load_dataset("coastalcph/lex_glue", "case_hold")
print(dataset_casehold['validation'][0])
print(len(dataset_casehold['validation']))

Here, we choose one of the datasets above:

In [ ]:
def get_label_key_ids(task_name: str, label_index: int|list[int], dataset_features) -> str | list[str]:
  dataset_labels = extract_labels(LEXGLUE_TASKS_DICT[task_name]["task_type"], dataset_features)
  if (type(label_index) == list):
    result = []
    for l in label_index:
      result.append(dataset_labels[l])
    return result

  return dataset_labels[label_index]


def get_actual_readable_label_name(task_name: str, label_index: int|list[int], dataset_features) -> str | list[str]:
  actual_label = get_label_key_ids(task_name, label_index, dataset_features)
  if (LEXGLUE_TASKS_DICT[task_name]["label_names"] == None):
    return actual_label
  else:
    readable_labels = LEXGLUE_TASKS_DICT[task_name]["label_names"]
    dataset_labels = extract_labels(LEXGLUE_TASKS_DICT[task_name]["task_type"], dataset_features)
    label_dict = build_label_map(dataset_labels, readable_labels)
    if (type(actual_label) == list):
      result = []
      for l in actual_label:
        result.append(label_dict[l])
      return result

    return label_dict[actual_label]

Adding a caching mechanism functions for caching repetitive llm calls:

In [ ]:
from pathlib import Path

CACHE_DIR = Path("./thesis_rlm_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [327]:
import json
import hashlib
from pathlib import Path
from typing import Any

CACHE_DIR = Path("./thesis_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def get_cache_file(
    task_name: str,
    split_name: str,
    method_name: str,
    model_name: str,
    temperature: float,
) -> Path:
    safe_model_name = model_name.replace("/", "_")
    filename = (
        f"{task_name}-{split_name}-{method_name}-"
        f"{safe_model_name}-temp{temperature}.json"
    )
    cache_folder = "rlm" if method_name == "rlm" else "vanilla"
    print(f"Cache file path: {CACHE_DIR / cache_folder / filename}")
    return CACHE_DIR / cache_folder / filename


def load_cache(cache_file: Path) -> dict[str, Any]:
    if not cache_file.exists():
        return {}

    try:
        with cache_file.open("r", encoding="utf-8") as file:
            return json.load(file)
    except (json.JSONDecodeError, OSError):
        return {}


def save_cache(cache_file: Path, cache: dict[str, Any]) -> None:
    temp_file = cache_file.with_suffix(".tmp")

    with temp_file.open("w", encoding="utf-8") as file:
        json.dump(cache, file, ensure_ascii=False, indent=2)

    temp_file.replace(cache_file)


def get_task_text_hash(prompt: str) -> str:
    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:16]

The configuration for the current dataset in evaluation:

In [220]:
# Configuration

split_name = "validation"
method_name = "rlm"
model_name = "gemini-2.5-flash"
current_dataset = dataset_eurlex
current_dataset_features = current_dataset[split_name].features
current_task_name = "EURLEX"
max_sample = 100

In [ ]:
# For fixing the missed samples

task_type = LEXGLUE_TASKS_DICT[current_task_name]['task_type']
label_attr = get_data_item_label_attr_name(task_type=task_type)
cache_file = get_cache_file(
    task_name=current_task_name,
    split_name=split_name,
    method_name=method_name,
    model_name=model_name,
    temperature=0,
)

cache = load_cache(cache_file)
index =  9
data_item = current_dataset[split_name][index]

prompt = generate_prompt(
task_name=current_task_name,
data_features=current_dataset_features,
data_item=data_item
)

cache_key = str(index)
text_hash = get_task_text_hash(prompt)

cached_item = cache.get(cache_key)


result = rlm.completion(prompt=prompt)
prediction = result.response

label_index = data_item[label_attr]
if (task_type == TASK_TYPE["MULTIPLE_CHOICE_QA"]):
    actual_res = label_index
else:
    actual_res = get_actual_readable_label_name(
    task_name=current_task_name,
    label_index=label_index,
    dataset_features=current_dataset_features
)

cache[cache_key] = {
        "prediction": prediction,
        "actual_label": actual_res,
        "text_hash": text_hash,
}

save_cache(cache_file, cache)


Cache file path: thesis_cache/EURLEX-validation-rlm-gemini-2.5-flash-temp0.json


──────────────────────────────────────────────────  Iteration 1  ──────────────────────────────────────────────────

╭─ ◇ LLM Response  (1.35s) ───────────────────────────────────────────────────────────────────────────────────────╮
│ ```repl                                                                                                         │
│ print(f"Context length: {len(context)} characters.")                                                            │
│ print(f"Context sample (first 500 characters):\n{context[:500]}")                                               │
│ ```                                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│ ~11 words                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ ▸ Code Execution  (0.000s) ────────────────────────────────────────────────────────────────────────────────────╮
│ Code:                                                                                                           │
│ print(f"Context length: {len(context)} characters.")                                                            │
│ print(f"Context sample (first 500 characters):\n{context[:500]}")                                               │
│                                                                                                                 │
│ Output:                                                                                                         │
│ Context length: 4762 characters.                                                                                │
│ Context sample (first 500 characters):                                                                          │
│ You are performing a multi-label legal document classification task.                                            │
│                                                                                                                 │
│ Task: Assign all applicable EuroVoc legal concepts to this European Union legal document.                       │
│                                                                                                                 │
│ Select every applicable label from the candidate labels below.                                                  │
│ Zero, one, or multiple labels may apply.                                                                        │
│                                                                                                                 │
│ Candidate labels: ['political framework', 'politics and public safety', 'executive power and public service',   │
│ 'international affairs', 'cooperation policy', 'international security', 'defence', 'EU institutions and        │
│ European civil serv                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

──────────────────────────────────────────────────  Iteration 2  ──────────────────────────────────────────────────

...

╭─ ★ Final Answer ────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ['European Union law', 'trade policy', 'tariff policy', 'trade', 'international trade', 'prices', 'taxation',  │
│  'economic policy', 'agricultural policy', 'agricultural structures and production', 'plant product',           │
│  'processed agricultural produce', 'agri-foodstuffs', 'Europe']                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  Iterations       3                                                   
                                              Total Time       21.77s                                              
                                              Input Tokens     6,907                                               
                                              Output Tokens    966     

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

['100163', '100168', '100169', '100170', '100171', '100172', '100173', '100174', '100175', '100176', '100177', '100179', '100180', '100183', '100184', '100185', '100186', '100187', '100189', '100190', '100191', '100192', '100193', '100194', '100195', '100196', '100197', '100198', '100199', '100200', '100201', '100202', '100204', '100205', '100206', '100207', '100212', '100214', '100215', '100220', '100221', '100222', '100223', '100224', '100226', '100227', '100229', '100230', '100231', '100232', '100233', '100234', '100235', '100237', '100238', '100239', '100240', '100241', '100242', '100243', '100244', '100245', '100246', '100247', '100248', '100249', '100250', '100252', '100253', '100254', '100255', '100256', '100257', '100258', '100259', '100260', '100261', '100262', '100263', '100264', '100265', '100266', '100268', '100269', '100270', '100271', '100272', '100273', '100274', '100275', '100276', '100277', '100278', '100279', '100280', '100281', '100282', '100283', '100284', '100285']

In [ ]:
task_type = LEXGLUE_TASKS_DICT[current_task_name]['task_type']
label_attr = get_data_item_label_attr_name(task_type=task_type)
cache_file = get_cache_file(
    task_name=current_task_name,
    split_name=split_name,
    method_name=method_name,
    model_name=model_name,
    temperature=0,
)

cache = load_cache(cache_file)

for index, data_item in enumerate(current_dataset[split_name]):
  if max_sample > 0 and index >= max_sample:
    break
  prompt = generate_prompt(
    task_name=current_task_name,
    data_features=current_dataset_features,
    data_item=data_item
  )

  cache_key = str(index)
  text_hash = get_task_text_hash(prompt)

  cached_item = cache.get(cache_key)
  
  if (
        cached_item is not None
        and cached_item.get("text_hash") == text_hash
    ):
      prediction = cached_item["prediction"]
      actual_res = cached_item["actual_label"]
      source = "cache"
      print(f"Cache hit for sample index {index}. Using cached result.")

  else:
    result = rlm.completion(prompt=prompt)
    prediction = result.response

    label_index = data_item[label_attr]
    if (task_type == TASK_TYPE["MULTIPLE_CHOICE_QA"]):
      actual_res = label_index
    else:
      actual_res = get_actual_readable_label_name(
        task_name=current_task_name,
        label_index=label_index,
        dataset_features=current_dataset_features
      )
    
    cache[cache_key] = {
          "prediction": prediction,
          "actual_label": actual_res,
          "text_hash": text_hash,
    }

    save_cache(cache_file, cache)
    source = "API"
    print(f"Cache miss for sample index {index}. Using API result and saving to cache.")

  print(f"data row number {index + 1}.--------------------")
  print(f"Predicted: {prediction}, actual: {actual_res}")
  print("---------------------------")


Now, we move forward to calculating the accuracy:

In [253]:
from collections.abc import Sequence

from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer


def evaluate_predictions(
    actual_labels: Sequence,
    predicted_labels: Sequence,
    task_type: int,
    number_of_labels: int | None = None,
) -> dict[str, float]:
    if len(actual_labels) != len(predicted_labels):
        raise ValueError(
            "actual_labels and predicted_labels must contain "
            "the same number of samples."
        )

    if task_type in {TASK_TYPE["MULTI_CLASS"], TASK_TYPE["MULTIPLE_CHOICE_QA"]}:
        return {
            "accuracy": accuracy_score(
                actual_labels,
                predicted_labels,
            ),
            "macro_f1": f1_score(
                actual_labels,
                predicted_labels,
                average="macro",
                zero_division=0,
            ),
            "micro_f1": f1_score(
                actual_labels,
                predicted_labels,
                average="micro",
                zero_division=0,
            ),
        }

    if task_type == TASK_TYPE["MULTI_LABEL"]:
        if number_of_labels is None:
            raise ValueError(
                "number_of_labels is required for multi-label evaluation."
            )

        mlb = MultiLabelBinarizer(
            classes=list(range(number_of_labels))
        )

        y_true = mlb.fit_transform(actual_labels)
        y_pred = mlb.transform(predicted_labels)

        return {
            "exact_match_accuracy": accuracy_score(y_true, y_pred),
            "micro_f1": f1_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0,
            ),
            "macro_f1": f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),
        }

    raise ValueError(f"Unsupported task type: {task_type}")

In [330]:
# Configuration for accuracy
CACHE_FILE = "./thesis_cache/vanilla/CaseHOLD-validation-vanilla-gemini-2.5-flash-temp0.json"
current_task_name = "CaseHOLD"
TASK_KIND = LEXGLUE_TASKS_DICT[current_task_name]['task_type']

In [331]:
import json
from pathlib import Path
from typing import Any

from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer


def load_cache(cache_file: str) -> dict[str, dict[str, Any]]:
    path = Path(cache_file)

    if not path.exists():
        raise FileNotFoundError(f"Cache file not found: {path}")

    with path.open("r", encoding="utf-8") as file:
        cache = json.load(file)

    if not isinstance(cache, dict):
        raise ValueError("The cache must be a JSON object.")

    return cache


def sorted_cache_items(
    cache: dict[str, dict[str, Any]],
) -> list[tuple[str, dict[str, Any]]]:
    """
    Sort cache keys numerically: '2' comes before '10'.
    """

    return sorted(
        cache.items(),
        key=lambda pair: int(pair[0]),
    )

# for multi-class and multiple-choice QA
def evaluate_multi_class(
    cache: dict[str, dict[str, Any]],
) -> dict[str, float]:

    y_true = []
    y_pred = []

    for cache_key, item in sorted_cache_items(cache):
        if "actual_label" not in item or "prediction" not in item:
            print(f"Skipping cache item {cache_key}: missing values.")
            continue

        actual = item["actual_label"]
        prediction = item["prediction"]

        # Handle values such as 4 versus "4".
        if isinstance(actual, int):
            try:
                prediction = int(str(prediction).strip())
            except (TypeError, ValueError):
                print(
                    f"Skipping item {cache_key}: "
                    f"invalid numeric prediction {prediction!r}"
                )
                continue

        # Handle string labels and remove accidental whitespace.
        elif isinstance(actual, str):
            actual = actual.strip()
            prediction = str(prediction).strip()

        y_true.append(actual)
        y_pred.append(prediction)

    if not y_true:
        raise ValueError("No valid predictions were found.")

    return {
        "number_of_samples": len(y_true),
        "correct_predictions": int(
            accuracy_score(y_true, y_pred, normalize=False)
        ),
        "accuracy": accuracy_score(y_true, y_pred),
        "micro_f1": f1_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
    }


# Normalize one multi-label prediction or actual-label value.
def normalize_label_list(value: Any) -> list[str]:

    if value is None:
        return []

    if not isinstance(value, list):
        raise TypeError(
            f"Expected a list for multi-label data, got {type(value).__name__}"
        )

    return sorted(
        {
            str(label).strip()
            for label in value
            if str(label).strip()
        }
    )


# for multi label classification
def evaluate_multi_label(
    cache: dict[str, dict[str, Any]],
) -> dict[str, float]:
    y_true_labels = []
    y_pred_labels = []

    for cache_key, item in sorted_cache_items(cache):
        if "actual_label" not in item or "prediction" not in item:
            print(f"Skipping cache item {cache_key}: missing values.")
            continue

        try:
            actual = normalize_label_list(item["actual_label"])
            prediction = normalize_label_list(item["prediction"])
        except TypeError as error:
            print(f"Skipping item {cache_key}: {error}")
            continue

        y_true_labels.append(actual)
        y_pred_labels.append(prediction)

    if not y_true_labels:
        raise ValueError("No valid multi-label predictions were found.")

    # Include labels appearing in either actual values or predictions.
    # This ensures hallucinated labels count as false positives.
    all_labels = sorted(
        {
            label
            for sample in y_true_labels + y_pred_labels
            for label in sample
        }
    )

    mlb = MultiLabelBinarizer(classes=all_labels)
    mlb.fit([all_labels])

    y_true = mlb.transform(y_true_labels)
    y_pred = mlb.transform(y_pred_labels)

    exact_matches = sum(
        set(actual) == set(prediction)
        for actual, prediction in zip(
            y_true_labels,
            y_pred_labels,
        )
    )

    return {
        "number_of_samples": len(y_true_labels),
        "number_of_labels": len(all_labels),

        # All labels for a document must match exactly.
        "exact_match_count": exact_matches,
        "exact_match_accuracy": accuracy_score(y_true, y_pred),

        # Usually the most important value for multi-label LexGLUE tasks.
        "micro_f1": f1_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
        "samples_f1": f1_score(
            y_true,
            y_pred,
            average="samples",
            zero_division=0,
        ),
    }


def print_results(results: dict[str, float]) -> None:
    print("\nEvaluation results")
    print("------------------")

    for metric, value in results.items():
        readable_name = metric.replace("_", " ").title()

        if isinstance(value, float):
            print(f"{readable_name}: {value:.4f}")

            if "accuracy" in metric or "f1" in metric:
                print(f"  Percentage: {value * 100:.2f}%")
        else:
            print(f"{readable_name}: {value}")


cache = load_cache(CACHE_FILE)

if TASK_KIND == TASK_TYPE["MULTI_LABEL"]:
    results = evaluate_multi_label(cache)

elif TASK_KIND == TASK_TYPE["MULTI_CLASS"] or TASK_KIND == TASK_TYPE["MULTIPLE_CHOICE_QA"]:
    results = evaluate_multi_class(cache)

else:
    raise ValueError(
        "TASK_KIND must be 'multi_label', "
        "'multi_class', or 'multiple_choice'."
    )

print_results(results)


Evaluation results
------------------
Number Of Samples: 100
Correct Predictions: 79
Accuracy: 0.7900
  Percentage: 79.00%
Micro F1: 0.7900
  Percentage: 79.00%
Macro F1: 0.7882
  Percentage: 78.82%
Weighted F1: 0.7910
  Percentage: 79.10%


Now, we use vanilla LLM for evaluating the benchmark:


In [230]:
import os

from google import genai
from google.genai import types


client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

In [325]:
# Configuration

split_name = "validation"
method_name = "vanilla"
model_name = "gemini-2.5-flash"
current_dataset = dataset_casehold
current_dataset_features = current_dataset[split_name].features
current_task_name = "CaseHOLD"
max_sample = 100

In [328]:
import time


task_type = LEXGLUE_TASKS_DICT[current_task_name]["task_type"]
label_attr = get_data_item_label_attr_name(task_type=task_type)


cache_file = get_cache_file(
    task_name=current_task_name,
    split_name=split_name,
    method_name=method_name,
    model_name=model_name,
    temperature=0,
)

cache = load_cache(cache_file)

for index, data_item in enumerate(current_dataset[split_name]):
    if max_sample > 0 and index >= max_sample:
        break

    prompt = generate_prompt(
        task_name=current_task_name,
        data_features=current_dataset_features,
        data_item=data_item,
    )

    cache_key = str(index)
    text_hash = get_task_text_hash(prompt)

    cached_item = cache.get(cache_key)

    if (
        cached_item is not None
        and cached_item.get("text_hash") == text_hash
    ):
        prediction = cached_item["prediction"]
        actual_res = cached_item["actual_label"]
        source = "cache"

        print(
            f"Cache hit for sample index {index}. "
            "Using cached result."
        )

    else:
        start_time = time.perf_counter()

        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0,
                ),
            )
        except Exception as error:
            print(
                f"Gemini request failed for sample {index}: "
                f"{type(error).__name__}: {error}"
            )
            continue

        latency_seconds = time.perf_counter() - start_time

        if not response.text:
            print(
                f"Gemini returned no text for sample {index}. "
                "Skipping this sample."
            )
            continue

        prediction = response.text.strip()

        label_index = data_item[label_attr]

        if task_type == TASK_TYPE["MULTIPLE_CHOICE_QA"]:
            actual_res = label_index
        else:
            actual_res = get_actual_readable_label_name(
                task_name=current_task_name,
                label_index=label_index,
                dataset_features=current_dataset_features,
            )

        usage = getattr(response, "usage_metadata", None)

        cache_item = {
            "prediction": prediction,
            "actual_label": actual_res,
            "text_hash": text_hash,
            "method": method_name,
            "model": model_name,
            "temperature": 0,
            "latency_seconds": round(latency_seconds, 4),
        }

        if usage is not None:
            cache_item["usage"] = {
                "prompt_token_count": getattr(
                    usage,
                    "prompt_token_count",
                    None,
                ),
                "candidates_token_count": getattr(
                    usage,
                    "candidates_token_count",
                    None,
                ),
                "thoughts_token_count": getattr(
                    usage,
                    "thoughts_token_count",
                    None,
                ),
                "total_token_count": getattr(
                    usage,
                    "total_token_count",
                    None,
                ),
            }

        cache[cache_key] = cache_item
        save_cache(cache_file, cache)

        source = "API"

        print(
            f"Cache miss for sample index {index}. "
            "Using vanilla Gemini result and saving to cache."
        )

    print(f"Data row number {index + 1} --------------------")
    print(f"Source: {source}")
    print(f"Predicted: {prediction}")
    print(f"Actual: {actual_res}")

    if source == "API":
        print(f"Latency: {latency_seconds:.2f} seconds")

        if usage is not None:
            print(
                "Token usage:",
                cache[cache_key]["usage"],
            )

    print("-----------------------------------------------")

Cache file path: thesis_cache/vanilla/CaseHOLD-validation-vanilla-gemini-2.5-flash-temp0.json
Cache miss for sample index 0. Using vanilla Gemini result and saving to cache.
Data row number 1 --------------------
Source: API
Predicted: 3
Actual: 3
Latency: 3.67 seconds
Token usage: {'prompt_token_count': 419, 'candidates_token_count': 1, 'thoughts_token_count': 617, 'total_token_count': 1037}
-----------------------------------------------
Cache miss for sample index 1. Using vanilla Gemini result and saving to cache.
Data row number 2 --------------------
Source: API
Predicted: 0
Actual: 0
Latency: 4.90 seconds
Token usage: {'prompt_token_count': 486, 'candidates_token_count': 1, 'thoughts_token_count': 876, 'total_token_count': 1363}
-----------------------------------------------
Cache miss for sample index 2. Using vanilla Gemini result and saving to cache.
Data row number 3 --------------------
Source: API
Predicted: 4
Actual: 4
Latency: 3.64 seconds
Token usage: {'prompt_token_co

Fix missed items due to the lack of handling asynchronous calls

In [289]:
index =  87
data_item = current_dataset[split_name][index]
print(data_item)

{'text': ['8.  The first applicant was born in 1992, the second applicant in 1995, and the third applicant in 1993. They are female and reside in Latvia.', '9.  The relevant facts of the case as submitted by the parties and emerging from the documents furnished to the Court may be summarised as follows.', '10.  From a young age the applicants trained in modern pentathlon. In 2007 they enrolled in a State sports school in Riga.', '11.  According to their submission, the first and third applicants commenced training with the sports school’s coach O.B. in summer 2008 and the second applicant was coached by him during the summer of 2009.', '12.  In addition to the above-mentioned training, O.B. organised sports summer camps on the school’s campus, which the first and third applicants attended in 2008 and all the applicants attended in 2009.', '13.  At the time of the events in issue, the applicants were under eighteen years of age.', '14.  On 4 January 2010 the mother of the first and seco

In [329]:
import time


task_type = LEXGLUE_TASKS_DICT[current_task_name]["task_type"]
label_attr = get_data_item_label_attr_name(task_type=task_type)


cache_file = get_cache_file(
    task_name=current_task_name,
    split_name=split_name,
    method_name=method_name,
    model_name=model_name,
    temperature=0,
)

cache = load_cache(cache_file)
index =  93
data_item = current_dataset[split_name][index]

prompt = generate_prompt(
    task_name=current_task_name,
    data_features=current_dataset_features,
    data_item=data_item,
)

cache_key = str(index)
text_hash = get_task_text_hash(prompt)

cached_item = cache.get(cache_key)

if (
    cached_item is not None
    and cached_item.get("text_hash") == text_hash
):
    prediction = cached_item["prediction"]
    actual_res = cached_item["actual_label"]
    source = "cache"

    print(
        f"Cache hit for sample index {index}. "
        "Using cached result."
    )

else:
    start_time = time.perf_counter()

    try:
        response = client.models.generate_content(
            model=model_name,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0,
            ),
        )
        print(response)
    except Exception as error:
        print(
            f"Gemini request failed for sample {index}: "
            f"{type(error).__name__}: {error}"
        )

    latency_seconds = time.perf_counter() - start_time

    if not response.text:
        print(
            f"Gemini returned no text for sample {index}. "
            "Skipping this sample."
        )

    prediction = response.text.strip()

    label_index = data_item[label_attr]

    if task_type == TASK_TYPE["MULTIPLE_CHOICE_QA"]:
        actual_res = label_index
    else:
        actual_res = get_actual_readable_label_name(
            task_name=current_task_name,
            label_index=label_index,
            dataset_features=current_dataset_features,
        )

    usage = getattr(response, "usage_metadata", None)

    cache_item = {
        "prediction": prediction,
        "actual_label": actual_res,
        "text_hash": text_hash,
        "method": method_name,
        "model": model_name,
        "temperature": 0,
        "latency_seconds": round(latency_seconds, 4),
    }

    if usage is not None:
        cache_item["usage"] = {
            "prompt_token_count": getattr(
                usage,
                "prompt_token_count",
                None,
            ),
            "candidates_token_count": getattr(
                usage,
                "candidates_token_count",
                None,
            ),
            "thoughts_token_count": getattr(
                usage,
                "thoughts_token_count",
                None,
            ),
            "total_token_count": getattr(
                usage,
                "total_token_count",
                None,
            ),
        }

    cache[cache_key] = cache_item
    save_cache(cache_file, cache)

    source = "API"

    print(
        f"Cache miss for sample index {index}. "
        "Using vanilla Gemini result and saving to cache."
    )

print(f"Data row number {index + 1} --------------------")
print(f"Source: {source}")
print(f"Predicted: {prediction}")
print(f"Actual: {actual_res}")

if source == "API":
    print(f"Latency: {latency_seconds:.2f} seconds")

    if usage is not None:
        print(
            "Token usage:",
            cache[cache_key]["usage"],
        )

print("-----------------------------------------------")

Cache file path: thesis_cache/vanilla/CaseHOLD-validation-vanilla-gemini-2.5-flash-temp0.json
sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""The legal context discusses various cases related to "actual damages" under statutes like RESPA. The blank refers to `John-stone v. Bank of America, N.A, 173 F.Supp.2d 809, 815 (N.D.Ill.2001)`.

1.  **Analyze the surrounding context:**
    *   The preceding case (M.D.Ala.1999) discusses actual damages for correspondence, travel, and time away from work – all pecuniary losses.
    *   The subsequent citation to the *same* Johnstone case (at page 816) states that "plaintiff could recover for loss of time and inconvenience provided that she could establish actual pecuniary loss." This strongly suggests a focus on pecuniary losses for actual damages.
    *   The overall discussion is about what constitutes "actual damages."

2.  **Research the specific holding